# Geometric Transforms
stough 202-

1. [Homogeneous coordinates](#homogeneous)
1. [The affine family](#affine)
1. [Composing transforms](#compose)
1. [Warping an image](#warp)
1. [Interactive: rotate and scale](#interactive)
1. [Projective transforms](#projective)
1. [🔨 Your Turn: Rectify a Photo](#your-turn)
1. [🧠 Further Efforts](#further)

In [Coordinate Operations](./coord_ops.ipynb) we rotated and scaled an image with $2 \times 2$ matrices, using *inverse mapping*: for every pixel of the output, compute where it came from in the input, and interpolate there. That notebook ended with two loose threads.

- To rotate about the image's center rather than its corner, we had to subtract the center, apply the matrix, and add the center back. Shifting, or **translation**, is not something a $2 \times 2$ matrix can do, since multiplying any matrix by the origin $\langle 0, 0 \rangle$ gives back the origin.
- Rotation, scaling and shearing all keep parallel lines parallel. But take a photo of a sheet of paper from an angle and its parallel edges converge. That warp isn't a $2 \times 2$ matrix either.

In this notebook we tie up both threads with one idea, **homogeneous coordinates**, which turn translation, and then perspective, into matrix multiplication too.

In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
import numpy as np
from skimage import data
from skimage.transform import AffineTransform, ProjectiveTransform, warp
from ipywidgets import VBox, FloatSlider

# For importing from alternative directory sources
import sys
sys.path.insert(0, '../dip_utils')

from matrix_utils import arr_info
from vis_utils import vis_pair

&nbsp;

<a id='homogeneous'></a>
## Homogeneous Coordinates

The trick is to add a third coordinate that is always 1: the point $(x, y)$ becomes $\langle x, y, 1 \rangle$. Now a $3 \times 3$ matrix can translate:

\begin{equation*}
\begin{bmatrix} x' \\ y' \\ 1 \end{bmatrix} =
\begin{bmatrix} 1 & 0 & t_x \\ 0 & 1 & t_y \\ 0 & 0 & 1 \end{bmatrix}
\begin{bmatrix} x \\ y \\ 1 \end{bmatrix} =
\begin{bmatrix} x + t_x \\ y + t_y \\ 1 \end{bmatrix}
\end{equation*}

The constant 1 in the third coordinate multiplies the last column, which adds $t_x$ and $t_y$. Our old $2 \times 2$ matrices fit in the top-left corner, with zeros beside and below and a 1 in the corner. So rotation by $\theta$, for example, becomes

\begin{equation*}
\mathbf{R}(\theta) = \begin{bmatrix} \cos\theta & -\sin\theta & 0 \\ \sin\theta & \cos\theta & 0 \\ 0 & 0 & 1 \end{bmatrix}.
\end{equation*}

One change of convention from [coord_ops](./coord_ops.ipynb): there we wrote points as $(row, col)$, matching how we index an image. Here we'll write them as $(x, y) = (col, row)$, which is what scikit-image's transforms expect. $x$ runs to the right and $y$ runs *down*, because row numbers increase downward. So a positive rotation angle turns *clockwise* on screen.

Let's write the basic transforms as small functions that each return a $3 \times 3$ matrix.

In [ ]:
def translate(tx, ty):
    return np.array([[1, 0, tx],
                     [0, 1, ty],
                     [0, 0, 1]], dtype=float)

def rotate(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s, 0],
                     [s,  c, 0],
                     [0,  0, 1]])

def scale(sx, sy=None):
    sy = sx if sy is None else sy
    return np.array([[sx, 0, 0],
                     [0, sy, 0],
                     [0,  0, 1]], dtype=float)

def shear(k):
    # Shift each point horizontally in proportion to its y coordinate.
    return np.array([[1, k, 0],
                     [0, 1, 0],
                     [0, 0, 1]], dtype=float)

translate(10, -5)

To see what these do, we'll transform a shape rather than an image: the outline of a letter F, whose lack of symmetry makes it easy to tell whether it's been rotated, flipped, or slanted. Its corners are stored as the columns of a $3 \times N$ matrix in homogeneous coordinates, so transforming every corner at once is a single matrix product, just as with `coords` in [coord_ops](./coord_ops.ipynb).

In [ ]:
# Corners of the letter F, as (x, y) pairs, going around its outline.
F = np.array([[0, 0], [3, 0], [3, 1], [1, 1], [1, 2], [2.5, 2], [2.5, 3], [1, 3], [1, 5], [0, 5], [0, 0]]).T
F = np.vstack([F, np.ones(F.shape[1])])   # add the row of 1s: homogeneous coordinates
print(arr_info(F))

def show_shapes(ax, shapes, title):
    '''Draw each 3xN homogeneous point list as a closed outline, y axis pointing down like an image.'''
    for P, style in zip(shapes, ['k--', 'C0-', 'C1-']):
        ax.plot(P[0], P[1], style, lw=2)
    ax.set_title(title)
    ax.set_aspect('equal')
    ax.set_xlim(-6, 9)
    ax.set_ylim(9, -6)   # rows increase downward
    ax.grid(alpha=0.3)

&nbsp;

<a id='affine'></a>
## The Affine Family

Any $3 \times 3$ matrix whose last row is $[0, 0, 1]$ is an **affine transform**:

\begin{equation*}
\begin{bmatrix} x' \\ y' \\ 1 \end{bmatrix} =
\begin{bmatrix} a & b & t_x \\ c & d & t_y \\ 0 & 0 & 1 \end{bmatrix}
\begin{bmatrix} x \\ y \\ 1 \end{bmatrix}
\end{equation*}

It has six free numbers: a $2 \times 2$ linear part ($a, b, c, d$) and a translation ($t_x, t_y$). Below, the original F is dashed and the transformed F is solid.

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(9, 6))
examples = [(translate(4, 2),                  'translate (4, 2)'),
            (rotate(np.pi / 6),                r'rotate $30°$'),
            (scale(1.5, 0.6),                  'scale (1.5, 0.6)'),
            (shear(0.6),                       'shear 0.6'),
            (scale(-1, 1),                     'reflect: scale (-1, 1)'),
            (np.array([[1.2, 0.5, 2], [-0.4, 0.9, 1], [0, 0, 1]]), 'a general affine')]
for a, (M, title) in zip(ax.flat, examples):
    show_shapes(a, [F, M @ F], title)
plt.tight_layout()

Each panel is one matrix applied to every corner of the F. A few things to notice:

- **Translation** slides the shape without turning or resizing it.
- **Rotation** turns it about the *origin*, the top-left corner of the grid at $(0, 0)$, not about the F's own center. Since $y$ points down, a positive angle turns clockwise.
- **Scaling** by different amounts in $x$ and $y$ squashes the F. A negative scale flips it: that's a **reflection**.
- **Shear** slides each point sideways in proportion to its height, the way a deck of cards slants when you push it sideways.
- The **general affine** does all of these at once.

And here's what all affine transforms have in common: straight lines stay straight, and *parallel lines stay parallel*. The F's two arms stay parallel to each other in every panel, even when they're no longer horizontal. Lengths and angles can change, but parallelism can't. Keep that in mind for later.

&nbsp;

<a id='compose'></a>
## Composing Transforms

The payoff of making everything a matrix is that a *sequence* of transforms is just a product of matrices. If we first apply $\mathbf{A}$ and then $\mathbf{B}$ to a point $\mathbf{p}$, we get $\mathbf{B}(\mathbf{A}\mathbf{p}) = (\mathbf{B}\mathbf{A})\mathbf{p}$. So the single matrix $\mathbf{B}\mathbf{A}$ does both. Note the order: the transform applied *first* is written on the *right*.

Matrix multiplication isn't commutative, and neither are transforms. Rotating and then translating is not the same as translating and then rotating:

In [ ]:
R = rotate(np.pi / 4)
T = translate(4, 0)

fig, ax = plt.subplots(1, 2, figsize=(8, 4))
show_shapes(ax[0], [F, R @ F, T @ R @ F], 'rotate, then translate:  T @ R')
show_shapes(ax[1], [F, T @ F, R @ T @ F], 'translate, then rotate:  R @ T')
plt.tight_layout()

In each panel the dashed F is the original, blue is after the first step, and orange is after both. On the left, the F rotates about the origin and then slides 4 units to the right. On the right, it first slides right, and then the rotation swings it, along with everything else, about the origin, so it ends up below and to the left of the blue F: rotated by the same angle, but in a different place.

This also explains the rotation about the center from [coord_ops](./coord_ops.ipynb#center). To rotate about a point $\mathbf{c}$: translate $\mathbf{c}$ to the origin, rotate, and translate back. As one matrix, that's

\begin{equation*}
\mathbf{M} = \mathbf{T}(\mathbf{c})\,\mathbf{R}(\theta)\,\mathbf{T}(-\mathbf{c}),
\end{equation*}

read right to left. Let's check with the center of the F.

In [ ]:
cx, cy = 1.5, 2.5      # roughly the middle of the F
M = translate(cx, cy) @ rotate(np.pi / 2) @ translate(-cx, -cy)
print(np.round(M, 3))

fig, ax = plt.subplots(figsize=(4, 4))
show_shapes(ax, [F, M @ F], r'rotate $90°$ about (1.5, 2.5)')
ax.plot(cx, cy, 'r+', ms=12, mew=2);

Three matrices multiplied into one. The result is still affine (its last row is still $[0, 0, 1]$), and the F turns in place about the red cross. Look at the printed matrix: its top-left $2 \times 2$ block is just the $90°$ rotation, and the last column is the translation that keeps the center fixed. However long a chain of affine transforms gets, it always collapses to a single affine matrix, so warping an image by the whole chain costs no more than warping by one transform.

&nbsp;

<a id='warp'></a>
## Warping an Image

Now let's apply transforms to an image. We could reuse the inverse-mapping code from [coord_ops](./coord_ops.ipynb#inverse), but scikit-image's [`warp`](https://scikit-image.org/docs/stable/api/skimage.transform.html#skimage.transform.warp) does exactly the same job: for every output pixel, it asks a *map* where that pixel comes from in the input, and interpolates there. [`AffineTransform`](https://scikit-image.org/docs/stable/api/skimage.transform.html#skimage.transform.AffineTransform) wraps a $3 \times 3$ matrix as a map.

There's one catch, and it trips up everyone at least once. Because `warp` works by inverse mapping, the map you give it must go from *output* coordinates back to *input* coordinates. That's the **inverse** of the transform you want to see. Every transform object has an `.inverse` for exactly this purpose.

In [ ]:
I = plt.imread('../dip_pics/peppers.png')[..., :3]
arr_info(I)

h, w = I.shape[:2]
center = (w / 2, h / 2)                     # (x, y)
M = translate(*center) @ rotate(np.pi / 12) @ scale(0.8) @ translate(-center[0], -center[1])
tform = AffineTransform(matrix=M)

J_right = warp(I, tform.inverse)            # correct: the map from output back to input
J_wrong = warp(I, tform)                    # the common mistake

fig, ax = plt.subplots(1, 3, figsize=(10, 3), sharex=True, sharey=True)
for a, img, title in zip(ax, [I, J_right, J_wrong],
                         ['Original', r'warp(I, tform.inverse)', r'warp(I, tform)']):
    a.imshow(img)
    a.set_title(title)
plt.tight_layout()

`M` was meant to shrink the peppers to 80% and turn them $15°$ clockwise about the center. With `tform.inverse`, that's what we get. With `tform` itself, `warp` applies the inverse of what we meant: the image grows by $1/0.8 = 1.25$ and turns counterclockwise. When a warp comes out backwards, this is the first thing to check.

Output pixels whose source lands outside the input are filled with 0 (black), just like our `valid` mask in coord_ops. And `warp` takes the same interpolation `order` as `map_coordinates` and `rescale`: 0 for nearest neighbor, 1 for bilinear (the default), 3 for bicubic.

&nbsp;

<a id='interactive'></a>
## Interactive: Rotate and Scale

Here are the same transforms with sliders. Each update builds one composite matrix, rotation and scaling about the image center, and calls `warp` once.

In [ ]:
plt.ioff()

angle_slider = FloatSlider(value=0, min=-180, max=180, step=5, description='angle (deg)')
scale_slider = FloatSlider(value=1, min=0.25, max=2, step=0.05, description='scale')

fig, ax = plt.subplots(figsize=(5, 3.5))
fig.canvas.header_visible = False
disp = ax.imshow(I)
ax.set_axis_off()

def update(change):
    M = (translate(*center) @ rotate(np.deg2rad(angle_slider.value))
         @ scale(scale_slider.value) @ translate(-center[0], -center[1]))
    disp.set_data(warp(I, AffineTransform(matrix=M).inverse))
    fig.canvas.draw()
    fig.canvas.flush_events()

angle_slider.observe(update, names='value')
scale_slider.observe(update, names='value')

VBox([angle_slider, scale_slider, fig.canvas])

&nbsp;

<a id='projective'></a>
## Projective Transforms

Affine transforms keep parallel lines parallel. A camera doesn't. When you photograph a flat surface from an angle, such as a page, a sign, or a tiled floor, the parallel edges that recede from you converge toward a vanishing point. Here's scikit-image's `text` sample image, a photo of handwriting on a board, taken at an angle.

In [ ]:
text = data.text()
arr_info(text)

# Four corners of a rectangular region on the board, as (x, y), going clockwise from top left.
# I read these off the interactive figure by hovering the mouse over each corner.
corners = np.array([[155, 15], [360, 95], [260, 130], [65, 40]], dtype=float)

plt.ion()
plt.figure(figsize=(7, 3))
plt.imshow(text, cmap='gray')
plt.plot(*np.vstack([corners, corners[:1]]).T, 'r-o')
plt.title('A rectangle on the board, seen at an angle');

The red outline marks a region that is a rectangle on the board itself, but in the photo it's a slanted quadrilateral, and its opposite sides aren't parallel. No affine transform can turn it back into a rectangle.

The fix is to free up the last row of the matrix. A **projective transform**, or **homography**, is a general $3 \times 3$ matrix

\begin{equation*}
\begin{bmatrix} u \\ v \\ w \end{bmatrix} =
\begin{bmatrix} a & b & c \\ d & e & f \\ g & h & 1 \end{bmatrix}
\begin{bmatrix} x \\ y \\ 1 \end{bmatrix},
\qquad
x' = \frac{u}{w} = \frac{ax + by + c}{gx + hy + 1}, \quad
y' = \frac{v}{w} = \frac{dx + ey + f}{gx + hy + 1}.
\end{equation*}

With $g$ and $h$ nonzero, the third coordinate $w$ is no longer 1, so we divide by it to get back to a point in the plane. That division makes points farther along the $(g, h)$ direction shrink toward each other, which is exactly the perspective effect. Straight lines still map to straight lines, but parallel lines can now meet. The bottom-right entry is fixed at 1 because scaling the whole matrix doesn't change $u/w$ and $v/w$, which leaves 8 free numbers.

**Estimating a homography from four points.** Where does the matrix come from? If we know where four points go, we know it. Each point pair $(x, y) \to (x', y')$ gives two equations. Multiplying out the denominator of the first,

\begin{equation*}
x'(gx + hy + 1) = ax + by + c
\quad\Longrightarrow\quad
ax + by + c - g\,x x' - h\,y x' = x',
\end{equation*}

which is *linear* in the 8 unknowns, and similarly for $y'$. Four point pairs give 8 linear equations in 8 unknowns: a system we can solve with `np.linalg.solve`. Let's map our four corners to the corners of a $300 \times 100$ rectangle.

In [ ]:
W, H = 300, 100
rect = np.array([[0, 0], [W - 1, 0], [W - 1, H - 1], [0, H - 1]], dtype=float)

def homography_from_4_points(src, dst):
    '''Solve for the 3x3 projective matrix taking each src point to its dst point.'''
    A, b = [], []
    for (x, y), (xp, yp) in zip(src, dst):
        A.append([x, y, 1, 0, 0, 0, -x * xp, -y * xp]);  b.append(xp)
        A.append([0, 0, 0, x, y, 1, -x * yp, -y * yp]);  b.append(yp)
    params = np.linalg.solve(np.array(A), np.array(b))
    return np.append(params, 1).reshape(3, 3)

Hmat = homography_from_4_points(corners, rect)
print(np.round(Hmat, 4))

# scikit-image can estimate it too (from any number of points >= 4, by least squares).
tform = ProjectiveTransform()
tform.estimate(corners, rect)
print('same as scikit-image:', np.allclose(Hmat, tform.params / tform.params[2, 2]))

Our 8-by-8 solve and scikit-image agree. Notice that $g$ and $h$, the first two entries of the bottom row, are small but not zero: that's the perspective.

Now we rectify. The homography maps the photo to the rectangle, so, just as with the affine warp, we hand `warp` its inverse, and ask for an output the size of the rectangle.

In [ ]:
rectified = warp(text, ProjectiveTransform(Hmat).inverse, output_shape=(H, W))

fig, ax = plt.subplots(1, 2, figsize=(10, 2.8), gridspec_kw={'width_ratios': [448, 300]})
ax[0].imshow(text, cmap='gray')
ax[0].plot(*np.vstack([corners, corners[:1]]).T, 'r-')
ax[0].set_title('Photo')
ax[1].imshow(rectified, cmap='gray')
ax[1].set_title('Rectified region')
plt.tight_layout()

The slanted region now reads as if we were looking straight at the board: the handwritten expression runs straight across, upright and evenly sized. This is how document-scanning apps flatten a photo of a page, and how sports broadcasts paint lines onto a field.

Two caveats. First, the photo only tells us where the corners are, not the region's true shape, so we had to *choose* the $300 \times 100$ output. If we pick the wrong aspect ratio, the writing comes out stretched. Second, in a strongly slanted photo the far side of a surface covers fewer pixels than the near side, so the far end of the rectified image comes out blurrier: it's been interpolated from less data. Rectifying can't recover detail the camera never recorded.

&nbsp;

<a id='your-turn'></a>
## 🔨 Your Turn: Rectify a Photo

**Task:** Take a photo of something flat and rectangular, at an angle: a page, a poster, a whiteboard, a laptop screen, or the face of a building. Rectify it so that you seem to be looking straight at it. Show the photo with the four corners marked, next to the rectified result.

**Guidance:**
- Load your photo with `plt.imread` and check it with `arr_info`. Shrink a phone photo first (e.g. with `skimage.transform.rescale`), since a 12-megapixel warp is slow; remember to find the corners on the shrunken image.
- To find the corner coordinates, display the image in an interactive (`%matplotlib widget`) figure and hover over each corner: the $(x, y)$ position appears in the figure's toolbar. Keep the corners in the same order (say clockwise from the top left) in both point lists.
- You need to choose the output rectangle's size. If you know the object's real shape (a letter-size page is $8.5 \times 11$ inches), use its aspect ratio. If you don't, how could you estimate it from the photo?
- If the result comes out mirrored, rotated, or as a strange sliver, check the order of your corners and whether you passed `warp` the transform or its inverse.
- For a color photo, `warp` handles all three channels at once.

&nbsp;

<a id='further'></a>
## 🧠 Further Efforts

1. **Which transforms commute?** We saw that rotation and translation don't commute. Which pairs of the basic transforms do: two rotations? Two translations? A rotation and a *uniform* scaling? A rotation and a *non-uniform* scaling? Predict each one first, and then check by comparing the matrix products.  
   *Guidance:* `np.allclose(A @ B, B @ A)` checks it numerically. For the ones that don't commute, draw both orders with `show_shapes` to see the difference.
1. **Affine from three points.** A homography needs four point pairs to pin down. How many does an affine transform need, and why? Write `affine_from_3_points(src, dst)` in the style of `homography_from_4_points`, and check it against `AffineTransform().estimate`.  
   *Guidance:* Count the unknowns and the equations each point pair provides. What happens when your three points lie on a line?
1. **Image registration.** Warp the peppers image with an affine transform that you keep secret (put it in a variable you don't look at). Then recover the transform by finding three or more matching points in the two images, estimate it, and undo the warp. How close do you get back to the original?  
   *Guidance:* Choose points that are easy to locate precisely in both images, such as the tips of stems. With more than three pairs, `estimate` finds the least-squares fit, which averages out your clicking errors. Measure the result with the mean absolute difference from the original, ignoring the black border.